# Value Learning Sigma Sweep Plots

Put this notebook in the sweep root, e.g. `.../slurm_experiments/value_learning_sigma_sweep/`, so it can find `runs/{dec,c}/c99s99/w32/lr.../sa.../sb.../<timestamp>/metrics.csv`.

It plots `greedy_team_rps` over training step for each learning rate, with the heuristic shown as a flat line using the mean of its last 10 evals.

In [ ]:
from __future__ import annotations

import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 80)

# If the notebook is inside the sweep root, Path.cwd() is enough.
# Otherwise, set RUN_ROOT manually to the folder that contains runs/.
RUN_ROOT = Path.cwd()
if not (RUN_ROOT / "runs").exists():
    candidates = [
        Path("slurm_experiments/value_learning_sigma_sweep"),
        Path("systematic_debug_report/slurm_experiments/value_learning_sigma_sweep"),
    ]
    for candidate in candidates:
        if (candidate / "runs").exists():
            RUN_ROOT = candidate.resolve()
            break

OUT_DIR = RUN_ROOT / "plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)

METRIC = "greedy_team_rps"
LAST_N_HEURISTIC_EVALS = 10
SAVE_FIGURES = True

EXPECTED_LRS = [0.001, 0.003, 0.0003, 0.0001, 0.00005, 0.00008]
EXPECTED_SIGMA_AS = [0, 1, 2, 4]
EXPECTED_SIGMA_BS = [0, 1, 2, 4, 8]

LR_LABELS = {
    0.001: "lr=1e-3",
    0.003: "lr=3e-3",
    0.0003: "lr=3e-4",
    0.0001: "lr=1e-4",
    0.00005: "lr=5e-5",
    0.00008: "lr=8e-5",
}
LR_ORDER = EXPECTED_LRS
LR_COLORS = {
    0.001: "#1f77b4",
    0.003: "#ff7f0e",
    0.0003: "#2ca02c",
    0.0001: "#9467bd",
    0.00005: "#8c564b",
    0.00008: "#e377c2",
}

print(f"RUN_ROOT = {RUN_ROOT}")
print(f"OUT_DIR  = {OUT_DIR}")

In [ ]:
def parse_tagged_float(tag: str, prefix: str) -> float:
    if not tag.startswith(prefix):
        raise ValueError(f"Expected tag starting with {prefix!r}, got {tag!r}")
    raw = tag[len(prefix):].replace("p", ".")
    return float(raw)


def parse_metrics_path(path: Path, root: Path) -> dict[str, object]:
    rel = path.relative_to(root)
    parts = rel.parts
    runs_idx = parts.index("runs")
    learning_tag = parts[runs_idx + 1]
    structure_tag = parts[runs_idx + 2]
    width_tag = parts[runs_idx + 3]
    lr_tag = parts[runs_idx + 4]
    sigma_a_tag = parts[runs_idx + 5]
    sigma_b_tag = parts[runs_idx + 6]
    run_id = parts[runs_idx + 7] if len(parts) > runs_idx + 8 else path.parent.name

    learning_type = {"dec": "decentralized", "c": "centralized"}.get(learning_tag, learning_tag)
    return {
        "learning_tag": learning_tag,
        "learning_type": learning_type,
        "structure_tag": structure_tag,
        "width": int(width_tag.removeprefix("w")),
        "lr": parse_tagged_float(lr_tag, "lr"),
        "sigma_a": parse_tagged_float(sigma_a_tag, "sa"),
        "sigma_b": parse_tagged_float(sigma_b_tag, "sb"),
        "run_id": run_id,
        "metrics_path": str(path),
    }


def load_metrics(root: Path) -> pd.DataFrame:
    paths = sorted((root / "runs").rglob("metrics.csv"))
    rows: list[pd.DataFrame] = []
    skipped: list[tuple[Path, str]] = []

    for path in paths:
        try:
            meta = parse_metrics_path(path, root)
            df = pd.read_csv(path)
            if df.empty:
                skipped.append((path, "empty csv"))
                continue
            for key, value in meta.items():
                df[key] = value
            df["run_key"] = df[["learning_tag", "structure_tag", "width", "lr", "sigma_a", "sigma_b", "run_id"]].astype(str).agg("/".join, axis=1)
            rows.append(df)
        except Exception as exc:
            skipped.append((path, repr(exc)))

    if skipped:
        print(f"Skipped {len(skipped)} metrics files:")
        for path, reason in skipped[:10]:
            print(f"  {path}: {reason}")
        if len(skipped) > 10:
            print("  ...")

    if not rows:
        return pd.DataFrame()
    data = pd.concat(rows, ignore_index=True)
    data["step"] = pd.to_numeric(data["step"], errors="coerce")
    data[METRIC] = pd.to_numeric(data[METRIC], errors="coerce")
    return data.dropna(subset=["step", METRIC]).sort_values(["learning_type", "width", "sigma_a", "sigma_b", "lr", "run_id", "step"])


data = load_metrics(RUN_ROOT)
raw_n_rows = len(data)
raw_n_runs = data["metrics_path"].nunique() if not data.empty else 0

if not data.empty:
    expected_lr_mask = data["lr"].apply(lambda lr: any(math.isclose(float(lr), expected, rel_tol=0, abs_tol=1e-12) for expected in EXPECTED_LRS))
    ignored_lr_runs = (
        data.loc[~expected_lr_mask, ["lr", "metrics_path"]]
        .drop_duplicates()
        .sort_values(["lr", "metrics_path"])
    )
    data = data.loc[expected_lr_mask].copy()
else:
    ignored_lr_runs = pd.DataFrame(columns=["lr", "metrics_path"])

print(f"Loaded {raw_n_rows:,} metric rows from {raw_n_runs} runs before LR filtering")
print(f"Keeping {len(data):,} metric rows from {data['metrics_path'].nunique() if not data.empty else 0} runs with EXPECTED_LRS")
if not ignored_lr_runs.empty:
    ignored_path = OUT_DIR / "ignored_unexpected_lr_runs.csv"
    ignored_lr_runs.to_csv(ignored_path, index=False)
    print(f"Ignored {ignored_lr_runs['metrics_path'].nunique()} runs with lr outside EXPECTED_LRS; saved {ignored_path}")
    display(ignored_lr_runs.head(20))

data.head()

In [ ]:
if data.empty:
    raise RuntimeError(f"No metrics.csv files found under {RUN_ROOT / 'runs'}")

heuristic_team_cols = [c for c in data.columns if c.endswith("_team_rps") and c != "greedy_team_rps"]
if "nearest_team_rps" in heuristic_team_cols:
    HEURISTIC_COL = "nearest_team_rps"
elif heuristic_team_cols:
    HEURISTIC_COL = heuristic_team_cols[0]
else:
    raise RuntimeError("Could not find a heuristic *_team_rps column in metrics.csv")

for col in [HEURISTIC_COL, "td_loss_avg"]:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

summary = (
    data.groupby(["learning_type", "width", "sigma_a", "sigma_b", "lr", "run_id"], dropna=False)
    .agg(first_step=("step", "min"), last_step=("step", "max"), n_evals=("step", "count"), path=("metrics_path", "first"))
    .reset_index()
)

display(summary.sort_values(["learning_type", "width", "sigma_a", "sigma_b", "lr"]).head(20))

expected_index = pd.MultiIndex.from_product(
    [
        sorted(data["learning_type"].unique()),
        sorted(data["width"].unique()),
        EXPECTED_SIGMA_AS,
        EXPECTED_SIGMA_BS,
        EXPECTED_LRS,
    ],
    names=["learning_type", "width", "sigma_a", "sigma_b", "lr"],
)
expected = expected_index.to_frame(index=False)
present = summary[["learning_type", "width", "sigma_a", "sigma_b", "lr"]].drop_duplicates()
coverage = expected.merge(
    present.assign(present=True),
    on=["learning_type", "width", "sigma_a", "sigma_b", "lr"],
    how="left",
)
coverage["present"] = coverage["present"].fillna(False).astype(bool)
missing_expected = coverage[~coverage["present"]].copy()
missing_path = OUT_DIR / "missing_expected_runs.csv"
missing_expected.to_csv(missing_path, index=False)
print(f"Expected configs: {len(coverage)} | present: {int(coverage['present'].sum())} | missing: {len(missing_expected)}")
print(f"saved {missing_path}")
if not missing_expected.empty:
    display(missing_expected.head(30))

print(f"Heuristic column: {HEURISTIC_COL}")
print(f"Learning types: {sorted(data['learning_type'].unique())}")
print(f"Widths: {sorted(data['width'].unique())}")
print(f"Sigma_a values: {sorted(data['sigma_a'].unique())}")
print(f"Sigma_b values: {sorted(data['sigma_b'].unique())}")
print(f"LR values: {sorted(data['lr'].unique())}")

In [ ]:
def aggregate_curve(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    return (
        df.groupby(["lr", "step"], as_index=False)
        .agg(mean=(metric, "mean"), std=(metric, "std"), n=(metric, "count"))
        .sort_values(["lr", "step"])
    )


def last_n_heuristic_mean(df: pd.DataFrame, last_n: int = LAST_N_HEURISTIC_EVALS) -> float:
    last_rows = (
        df.sort_values("step")
        .groupby("run_key", group_keys=False)
        .tail(last_n)
    )
    return float(last_rows[HEURISTIC_COL].mean())


def format_lr(lr: float) -> str:
    for known, label in LR_LABELS.items():
        if math.isclose(lr, known, rel_tol=0, abs_tol=1e-12):
            return label
    return f"lr={lr:g}"


def plot_learning_grid(df: pd.DataFrame, learning_type: str, width: int, metric: str = METRIC) -> plt.Figure:
    sub = df[(df["learning_type"] == learning_type) & (df["width"] == width)].copy()
    if sub.empty:
        raise ValueError(f"No rows for learning_type={learning_type}, width={width}")

    sigma_as = sorted(sub["sigma_a"].unique())
    sigma_bs = sorted(sub["sigma_b"].unique())
    fig, axes = plt.subplots(
        len(sigma_as),
        len(sigma_bs),
        figsize=(4.1 * len(sigma_bs), 3.0 * len(sigma_as) + 0.9),
        sharex=True,
        sharey=False,
        squeeze=False,
    )

    for row_idx, sigma_a in enumerate(sigma_as):
        for col_idx, sigma_b in enumerate(sigma_bs):
            ax = axes[row_idx, col_idx]
            panel = sub[(sub["sigma_a"] == sigma_a) & (sub["sigma_b"] == sigma_b)]
            ax.set_title(f"sa={sigma_a:g}, sb={sigma_b:g}", fontsize=10)
            if panel.empty:
                ax.text(0.5, 0.5, "missing", transform=ax.transAxes, ha="center", va="center", color="0.45")
                continue

            heuristic_y = last_n_heuristic_mean(panel)
            x_min = float(panel["step"].min())
            x_max = float(panel["step"].max())
            ax.hlines(
                heuristic_y,
                x_min,
                x_max,
                colors="0.25",
                linestyles="--",
                linewidth=1.5,
                label=f"{HEURISTIC_COL.replace('_team_rps', '')} last-{LAST_N_HEURISTIC_EVALS} mean" if row_idx == 0 and col_idx == 0 else None,
            )

            curve = aggregate_curve(panel, metric)
            y_values = [heuristic_y]
            lrs = [lr for lr in LR_ORDER if lr in set(curve["lr"])]
            lrs += [lr for lr in sorted(curve["lr"].unique()) if lr not in lrs]
            for lr in lrs:
                lr_curve = curve[curve["lr"] == lr]
                color = LR_COLORS.get(lr)
                ax.plot(lr_curve["step"], lr_curve["mean"], label=format_lr(lr), color=color, linewidth=1.7)
                y_values.extend(lr_curve["mean"].dropna().astype(float).tolist())
                if lr_curve["n"].max() > 1:
                    y = lr_curve["mean"].to_numpy(dtype=float)
                    std = lr_curve["std"].fillna(0.0).to_numpy(dtype=float)
                    x = lr_curve["step"].to_numpy(dtype=float)
                    ax.fill_between(x, y - std, y + std, color=color, alpha=0.12, linewidth=0)
                    y_values.extend((y - std).tolist())
                    y_values.extend((y + std).tolist())

            finite_y = np.asarray([v for v in y_values if np.isfinite(v)], dtype=float)
            if finite_y.size:
                y_min = float(finite_y.min())
                y_max = float(finite_y.max())
                pad = max((y_max - y_min) * 0.08, 0.05)
                ax.set_ylim(y_min - pad, y_max + pad)

            ax.ticklabel_format(axis="x", style="sci", scilimits=(6, 6))
            if col_idx == 0:
                ax.set_ylabel(metric)
            if row_idx == len(sigma_as) - 1:
                ax.set_xlabel("step")

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.suptitle(f"{metric} by learning rate | {learning_type} | width={width}", y=0.995, fontsize=14)
    fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.965),
        ncol=min(4, max(1, len(labels))),
        frameon=False,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.91))
    return fig


for width in sorted(data["width"].unique()):
    for learning_type in sorted(data["learning_type"].unique()):
        fig = plot_learning_grid(data, learning_type=learning_type, width=width)
        if SAVE_FIGURES:
            output = OUT_DIR / f"{METRIC}_{learning_type}_w{width}.png"
            fig.savefig(output, dpi=180, bbox_inches="tight")
            print(f"saved {output}")
        plt.show()

## Last-10-Eval Summary

The table below computes the last-10-eval mean for `greedy_team_rps` and the heuristic for each run, then averages repeated run timestamps within each condition.

In [ ]:
run_last = (
    data.sort_values("step")
    .groupby("run_key", group_keys=False)
    .tail(LAST_N_HEURISTIC_EVALS)
)

last10_summary = (
    run_last.groupby(["learning_type", "width", "sigma_a", "sigma_b", "lr", "run_id"], as_index=False)
    .agg(
        greedy_last10_team_rps=(METRIC, "mean"),
        heuristic_last10_team_rps=(HEURISTIC_COL, "mean"),
        last_step=("step", "max"),
        n_last_rows=("step", "count"),
    )
)

condition_summary = (
    last10_summary.groupby(["learning_type", "width", "sigma_a", "sigma_b", "lr"], as_index=False)
    .agg(
        greedy_last10_team_rps=("greedy_last10_team_rps", "mean"),
        heuristic_last10_team_rps=("heuristic_last10_team_rps", "mean"),
        n_runs=("run_id", "nunique"),
        max_step=("last_step", "max"),
    )
)
condition_summary["greedy_minus_heuristic"] = condition_summary["greedy_last10_team_rps"] - condition_summary["heuristic_last10_team_rps"]
_eps = 1e-9
condition_summary["greedy_over_heuristic"] = np.where(
    condition_summary["heuristic_last10_team_rps"].abs() > _eps,
    condition_summary["greedy_last10_team_rps"] / condition_summary["heuristic_last10_team_rps"],
    np.nan,
)
condition_summary["greedy_relative_lift"] = np.where(
    condition_summary["heuristic_last10_team_rps"].abs() > _eps,
    condition_summary["greedy_minus_heuristic"] / condition_summary["heuristic_last10_team_rps"].abs(),
    np.nan,
)

summary_path = OUT_DIR / "last10_condition_summary.csv"
condition_summary.to_csv(summary_path, index=False)
print(f"saved {summary_path}")
display(condition_summary.sort_values(["learning_type", "width", "sigma_a", "sigma_b", "lr"]).head(30))

## Best LR by Sigma B

For each fixed `sigma_a`, this selects the best learning rate separately for each `(learning_type, sigma_b)` condition, using last-10-eval mean `greedy_team_rps`. The marker labels show which LR won at each point.

In [ ]:
BEST_LR_SIGMA_AS = [0, 1, 2, 4]
LEARNING_LABELS = {"decentralized": "Decentralized", "centralized": "Centralized"}
LEARNING_COLORS = {"decentralized": "#1f77b4", "centralized": "#d62728"}
BEST_LR_X_SCALE = "log_with_zero_proxy"
BEST_LR_PLOT_VALUE = "greedy_over_heuristic"  # raw, greedy_over_heuristic, greedy_relative_lift

best_lr_rows = (
    condition_summary.sort_values("greedy_last10_team_rps", ascending=False)
    .groupby(["learning_type", "width", "sigma_a", "sigma_b"], as_index=False)
    .first()
)
best_lr_rows = best_lr_rows.sort_values(["width", "sigma_a", "learning_type", "sigma_b"])

best_lr_path = OUT_DIR / "best_lr_by_sigma_b.csv"
best_lr_rows.to_csv(best_lr_path, index=False)
print(f"saved {best_lr_path}")
display(best_lr_rows.head(20))

def best_lr_plot_columns(mode: str) -> tuple[str, str, float | None, str, str]:
    if mode == "raw":
        return (
            "greedy_last10_team_rps",
            "heuristic_last10_team_rps",
            None,
            "best LR last-10 greedy_team_rps",
            "Best learning rate over sigma_b",
        )
    if mode == "greedy_over_heuristic":
        return (
            "greedy_over_heuristic",
            "heuristic_reference",
            1.0,
            "learned / heuristic last-10 team RPS",
            "Best learning rate normalized by heuristic",
        )
    if mode == "greedy_relative_lift":
        return (
            "greedy_relative_lift",
            "heuristic_reference",
            0.0,
            "(learned - heuristic) / abs(heuristic)",
            "Best learning rate relative lift over heuristic",
        )
    raise ValueError(f"Unknown BEST_LR_PLOT_VALUE={mode!r}")

def plot_best_lr_vs_sigma_b(best_df: pd.DataFrame, width: int, sigma_as: list[float] = BEST_LR_SIGMA_AS) -> plt.Figure:
    value_col, heuristic_col, heuristic_reference, y_label, title = best_lr_plot_columns(BEST_LR_PLOT_VALUE)
    fig, axes = plt.subplots(2, 2, figsize=(10.5, 7.2), sharex=True, squeeze=False)
    axes_flat = axes.ravel()

    positive_sigma_bs = sorted(x for x in best_df["sigma_b"].dropna().unique() if x > 0)
    zero_proxy = positive_sigma_bs[0] / 2 if positive_sigma_bs else 0.5

    def sigma_b_x(values):
        arr = np.asarray(values, dtype=float)
        return np.where(arr <= 0, zero_proxy, arr)

    for ax, sigma_a in zip(axes_flat, sigma_as):
        panel = best_df[(best_df["width"] == width) & np.isclose(best_df["sigma_a"], sigma_a)]
        ax.set_title(f"sigma_a={sigma_a:g}")
        if panel.empty:
            ax.text(0.5, 0.5, "missing", transform=ax.transAxes, ha="center", va="center", color="0.45")
            continue

        y_values = []
        learning_types = sorted(panel["learning_type"].unique())
        heuristic_source_type = "decentralized" if "decentralized" in learning_types else learning_types[0]
        for learning_type in learning_types:
            line = panel[panel["learning_type"] == learning_type].sort_values("sigma_b")
            color = LEARNING_COLORS.get(learning_type)
            label = LEARNING_LABELS.get(learning_type, learning_type)
            x = sigma_b_x(line["sigma_b"])
            ax.plot(
                x,
                line[value_col],
                marker="o",
                linewidth=2.0,
                color=color,
                label=f"{label} best LR",
            )
            y_values.extend(line[value_col].dropna().astype(float).tolist())

        heuristic_line = panel[panel["learning_type"] == heuristic_source_type].sort_values("sigma_b")
        heuristic_x = sigma_b_x(heuristic_line["sigma_b"])
        heuristic_y = (
            np.full(len(heuristic_line), heuristic_reference, dtype=float)
            if heuristic_reference is not None
            else heuristic_line[heuristic_col].to_numpy(dtype=float)
        )
        ax.plot(
            heuristic_x,
            heuristic_y,
            marker="x",
            linewidth=1.7,
            linestyle="--",
            color="0.25",
            alpha=0.8,
            label="Heuristic",
        )
        y_values.extend(pd.Series(heuristic_y).dropna().astype(float).tolist())

        finite_y = np.asarray([v for v in y_values if np.isfinite(v)], dtype=float)
        if finite_y.size:
            y_min = float(finite_y.min())
            y_max = float(finite_y.max())
            pad = max((y_max - y_min) * 0.12, 0.05)
            ax.set_ylim(y_min - pad, y_max + pad)
        sigma_b_ticks = sorted(panel["sigma_b"].unique())
        tick_positions = sigma_b_x(sigma_b_ticks)
        if BEST_LR_X_SCALE == "log_with_zero_proxy":
            ax.set_xscale("log", base=2)
        ax.set_xlabel("sigma_b")
        ax.set_ylabel(y_label)
        ax.set_xticks(tick_positions)
        ax.set_xticklabels([f"{x:g}" for x in sigma_b_ticks])

    for ax in axes_flat[len(sigma_as):]:
        ax.axis("off")

    handles, labels = axes_flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=max(1, len(labels)), frameon=False, bbox_to_anchor=(0.5, 0.98))
    fig.suptitle(f"{title} | width={width}", y=1.03, fontsize=14)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    return fig


for width in sorted(best_lr_rows["width"].unique()):
    fig = plot_best_lr_vs_sigma_b(best_lr_rows, width=width)
    if SAVE_FIGURES:
        output = OUT_DIR / f"best_lr_vs_sigma_b_{BEST_LR_PLOT_VALUE}_w{width}.png"
        fig.savefig(output, dpi=180, bbox_inches="tight")
        print(f"saved {output}")
    plt.show()


In [ ]:
def plot_last10_heatmaps(summary: pd.DataFrame, value_col: str = "greedy_last10_team_rps") -> None:
    for width in sorted(summary["width"].unique()):
        for learning_type in sorted(summary["learning_type"].unique()):
            sub = summary[(summary["width"] == width) & (summary["learning_type"] == learning_type)]
            if sub.empty:
                continue
            lrs = [lr for lr in LR_ORDER if lr in set(sub["lr"])]
            lrs += [lr for lr in sorted(sub["lr"].unique()) if lr not in lrs]
            fig, axes = plt.subplots(1, len(lrs), figsize=(4.4 * len(lrs), 3.8), squeeze=False)
            vmin = sub[value_col].min()
            vmax = sub[value_col].max()
            for ax, lr in zip(axes[0], lrs):
                panel = sub[sub["lr"] == lr]
                pivot = panel.pivot(index="sigma_a", columns="sigma_b", values=value_col).sort_index().sort_index(axis=1)
                im = ax.imshow(pivot.to_numpy(), origin="lower", aspect="auto", vmin=vmin, vmax=vmax, cmap="viridis")
                ax.set_title(format_lr(lr))
                ax.set_xlabel("sigma_b")
                ax.set_ylabel("sigma_a")
                ax.set_xticks(range(len(pivot.columns)), [f"{x:g}" for x in pivot.columns])
                ax.set_yticks(range(len(pivot.index)), [f"{x:g}" for x in pivot.index])
                for i, sigma_a in enumerate(pivot.index):
                    for j, sigma_b in enumerate(pivot.columns):
                        val = pivot.loc[sigma_a, sigma_b]
                        if pd.notna(val):
                            ax.text(j, i, f"{val:.2f}", ha="center", va="center", color="white", fontsize=8)
            fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.82, label=value_col)
            fig.suptitle(f"Last-10 mean {value_col} | {learning_type} | width={width}", y=1.05)
            fig.tight_layout()
            if SAVE_FIGURES:
                output = OUT_DIR / f"last10_{value_col}_{learning_type}_w{width}.png"
                fig.savefig(output, dpi=180, bbox_inches="tight")
                print(f"saved {output}")
            plt.show()


plot_last10_heatmaps(condition_summary, "greedy_last10_team_rps")
plot_last10_heatmaps(condition_summary, "greedy_minus_heuristic")

## One Run Reward-Vector Panels


In [ ]:
# Plot one randomly selected run per sigma_a, showing all task reward vectors and their team sums.
# Adjust VECTOR_PLOT_FILTER to compare specific experiment families.

VECTOR_PLOT_SIGMA_AS = [1, 2, 4]
VECTOR_PLOT_SIGMA_BS = None  # e.g. [0, 1, 2, 4]; None keeps all sigma_b values
VECTOR_PLOT_RANDOM_SEED = 0
VECTOR_PLOT_FILTER = {
    "learning_type": "decentralized",
    # Examples; uncomment as needed:
    # "reward_generation": "sampled_mean",
    # "positive_rewards_only": True,
    # "positive_rewards_only": False,
    # "reward_generation": "baseline_offset",
    # "require_no_negative_dominates_positive": True,
}


def load_reward_vectors_for_vector_plots(root: Path) -> pd.DataFrame:
    paths = sorted((root / "runs").rglob("reward_vectors.csv"))
    rows = []
    skipped = []
    for path in paths:
        try:
            df = pd.read_csv(path)
            if df.empty:
                skipped.append((path, "empty csv"))
                continue
            meta = parse_metrics_path(path, root).copy()
            meta["reward_vectors_path"] = meta.pop("metrics_path", str(path))
            meta["run_dir"] = str(path.parent)
            for key, value in meta.items():
                df[key] = value
            df["run_key"] = df["run_dir"].astype(str)
            rows.append(df)
        except Exception as exc:
            skipped.append((path, repr(exc)))

    if skipped:
        print(f"Skipped {len(skipped)} reward vector files")
        for path, reason in skipped[:10]:
            print(f"  {path}: {reason}")
        if len(skipped) > 10:
            print("  ...")

    if not rows:
        return pd.DataFrame()

    out = pd.concat(rows, ignore_index=True)
    for col in ["width", "lr", "sigma_a", "sigma_b", "baseline_team_sum_mean", "task_type", "reward_sum", "reward_mean", "reward_variance"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def boolish_equal(series: pd.Series, value: object) -> pd.Series:
    if isinstance(value, bool):
        return series.astype(str).str.lower().isin({str(value).lower(), "1" if value else "0"})
    return series == value


def apply_vector_plot_filter(df: pd.DataFrame, filters: dict[str, object]) -> pd.DataFrame:
    out = df.copy()
    for col, value in filters.items():
        if value is None or col not in out.columns:
            continue
        if isinstance(value, (list, tuple, set)):
            if pd.api.types.is_numeric_dtype(out[col]):
                mask = np.zeros(len(out), dtype=bool)
                for item in value:
                    mask |= np.isclose(out[col].astype(float), float(item), equal_nan=False)
                out = out[mask]
            else:
                out = out[out[col].isin(value)]
        elif isinstance(value, (int, float)) and pd.api.types.is_numeric_dtype(out[col]):
            out = out[np.isclose(out[col].astype(float), float(value), equal_nan=False)]
        else:
            out = out[boolish_equal(out[col], value)]
    return out


if "reward_vectors" not in globals() or reward_vectors.empty:
    reward_vectors = load_reward_vectors_for_vector_plots(RUN_ROOT)

if reward_vectors.empty:
    raise RuntimeError(f"No reward_vectors.csv files found under {RUN_ROOT / 'runs'}")

vector_agent_cols = sorted(
    [c for c in reward_vectors.columns if re.fullmatch(r"reward_agent_\d+", c)],
    key=lambda c: int(c.rsplit("_", 1)[1]),
)
for col in vector_agent_cols:
    reward_vectors[col] = pd.to_numeric(reward_vectors[col], errors="coerce")

vector_rows = apply_vector_plot_filter(reward_vectors, VECTOR_PLOT_FILTER)
if VECTOR_PLOT_SIGMA_AS is not None:
    sigma_a_mask = np.zeros(len(vector_rows), dtype=bool)
    for sigma_a in VECTOR_PLOT_SIGMA_AS:
        sigma_a_mask |= np.isclose(vector_rows["sigma_a"].astype(float), float(sigma_a), equal_nan=False)
    vector_rows = vector_rows[sigma_a_mask].copy()

if VECTOR_PLOT_SIGMA_BS is not None:
    sigma_b_mask = np.zeros(len(vector_rows), dtype=bool)
    for sigma_b in VECTOR_PLOT_SIGMA_BS:
        sigma_b_mask |= np.isclose(vector_rows["sigma_b"].astype(float), float(sigma_b), equal_nan=False)
    vector_rows = vector_rows[sigma_b_mask].copy()

if vector_rows.empty:
    raise RuntimeError("No reward vectors matched VECTOR_PLOT_FILTER / sigma_a / sigma_b settings")

run_meta_cols = [
    c for c in [
        "run_dir", "learning_type", "reward_target", "reward_generation",
        "positive_rewards_only", "require_no_negative_dominates_positive",
        "baseline_team_sum_mean", "deterministic_baseline_offsets",
        "structure_tag", "width", "lr", "sigma_a", "sigma_b", "run_id",
    ]
    if c in vector_rows.columns
]
run_meta = vector_rows[run_meta_cols].drop_duplicates().copy()

rng = np.random.default_rng(VECTOR_PLOT_RANDOM_SEED)
selected_run_dirs = []
for sigma_a in VECTOR_PLOT_SIGMA_AS:
    candidates = run_meta[np.isclose(run_meta["sigma_a"].astype(float), float(sigma_a), equal_nan=False)]
    if candidates.empty:
        print(f"No candidate runs for sigma_a={sigma_a}")
        continue
    selected_idx = int(rng.integers(0, len(candidates)))
    selected = candidates.iloc[selected_idx]
    selected_run_dirs.append(str(selected["run_dir"]))

print("Selected runs:")
for run_dir in selected_run_dirs:
    row = run_meta[run_meta["run_dir"].astype(str) == run_dir].iloc[0]
    bits = []
    for col in run_meta_cols:
        if col == "run_dir":
            continue
        bits.append(f"{col}={row[col]}")
    print(f"  {run_dir}\n    " + ", ".join(bits))


def plot_reward_vectors_for_run(run_df: pd.DataFrame) -> plt.Figure:
    run_df = run_df.sort_values("task_type").copy()
    matrix = run_df[vector_agent_cols].to_numpy(dtype=float)
    task_types = run_df["task_type"].astype(int).to_numpy()
    task_sums = matrix.sum(axis=1)
    n_negative_agents = np.sum(matrix < 0.0, axis=1)
    avg_n_negative_agents = float(np.nanmean(n_negative_agents))
    per_agent_task_variance = np.nanvar(matrix, axis=0)
    avg_agent_task_variance = float(np.nanmean(per_agent_task_variance))

    meta = run_df.iloc[0]
    sigma_a = float(meta["sigma_a"])
    sigma_b = float(meta["sigma_b"]) if "sigma_b" in run_df.columns else np.nan
    lr = float(meta["lr"]) if "lr" in run_df.columns and pd.notna(meta["lr"]) else np.nan

    y_min = float(np.nanmin(matrix))
    y_max = float(np.nanmax(matrix))
    y_pad = max((y_max - y_min) * 0.08, 0.05)

    fig = plt.figure(figsize=(16, 13.0))
    gs = fig.add_gridspec(6, 4, height_ratios=[1, 1, 1, 0.75, 0.7, 0.7], hspace=0.62, wspace=0.32)

    for idx, (task_type, values, task_sum) in enumerate(zip(task_types, matrix, task_sums)):
        ax = fig.add_subplot(gs[idx // 4, idx % 4])
        ax.bar(range(len(values)), values, color="#4c78a8", alpha=0.9)
        ax.axhline(0.0, color="0.2", linewidth=1.0)
        ax.set_ylim(y_min - y_pad, y_max + y_pad)
        ax.set_title(f"task {task_type} | sum={task_sum:.3g}", fontsize=10)
        ax.set_xlabel("agent")
        ax.set_ylabel("reward")
        ax.grid(True, axis="y", alpha=0.28)

    sum_ax = fig.add_subplot(gs[3, :])
    colors = np.where(task_sums >= 0.0, "#54a24b", "#e45756")
    sum_ax.bar(task_types, task_sums, color=colors, alpha=0.9)
    sum_ax.axhline(float(np.nanmean(task_sums)), color="0.2", linestyle="--", linewidth=1.2, label="mean sum")
    sum_ax.set_title(
        f"task sums | mean={np.nanmean(task_sums):.3g}, std={np.nanstd(task_sums):.3g}, "
        f"min={np.nanmin(task_sums):.3g}, max={np.nanmax(task_sums):.3g}",
        fontsize=11,
    )
    sum_ax.set_xlabel("task type")
    sum_ax.set_ylabel("sum_j reward")
    sum_ax.set_xticks(task_types)
    sum_ax.grid(True, axis="y", alpha=0.3)
    sum_ax.legend(loc="best", frameon=False)

    neg_ax = fig.add_subplot(gs[4, :])
    neg_ax.bar(task_types, n_negative_agents, color="#f58518", alpha=0.9)
    neg_ax.axhline(avg_n_negative_agents, color="0.2", linestyle="--", linewidth=1.2, label="mean negative agents")
    neg_ax.set_title(
        f"negative agents per task | mean={avg_n_negative_agents:.3g}, "
        f"min={np.nanmin(n_negative_agents):.0f}, max={np.nanmax(n_negative_agents):.0f}",
        fontsize=11,
    )
    neg_ax.set_xlabel("task type")
    neg_ax.set_ylabel("# agents < 0")
    neg_ax.set_xticks(task_types)
    neg_ax.set_ylim(0, len(vector_agent_cols))
    neg_ax.grid(True, axis="y", alpha=0.3)
    neg_ax.legend(loc="best", frameon=False)

    var_ax = fig.add_subplot(gs[5, :])
    agent_ids = np.arange(len(vector_agent_cols))
    var_ax.bar(agent_ids, per_agent_task_variance, color="#b279a2", alpha=0.9)
    var_ax.axhline(avg_agent_task_variance, color="0.2", linestyle="--", linewidth=1.2, label="mean agent variance")
    var_ax.set_title(
        f"per-agent reward variance across task types | "
        f"mean var={avg_agent_task_variance:.3g}, "
        f"min var={np.nanmin(per_agent_task_variance):.3g}, "
        f"max var={np.nanmax(per_agent_task_variance):.3g}",
        fontsize=11,
    )
    var_ax.set_xlabel("agent")
    var_ax.set_ylabel("Var over tasks")
    var_ax.set_xticks(agent_ids)
    var_ax.grid(True, axis="y", alpha=0.3)
    var_ax.legend(loc="best", frameon=False)

    title_bits = [
        f"sa={sigma_a:g}",
        f"sb={sigma_b:g}" if np.isfinite(sigma_b) else "sb=nan",
        f"lr={lr:.3g}" if np.isfinite(lr) else "lr=nan",
    ]
    for col in ["reward_generation", "baseline_team_sum_mean", "deterministic_baseline_offsets", "positive_rewards_only", "require_no_negative_dominates_positive", "reward_target"]:
        if col in run_df.columns:
            title_bits.append(f"{col}={meta[col]}")
    fig.suptitle("Reward vectors for one selected run | " + " | ".join(title_bits), y=0.995, fontsize=14)
    fig.tight_layout(rect=(0, 0, 1, 0.97))

    if SAVE_FIGURES:
        output = OUT_DIR / f"reward_vectors_one_run_sa{sigma_a:g}.png"
        fig.savefig(output, dpi=180, bbox_inches="tight")
        print(f"saved {output}")
    return fig


for run_dir in selected_run_dirs:
    fig = plot_reward_vectors_for_run(vector_rows[vector_rows["run_dir"].astype(str) == run_dir])
    plt.show()
